# Reproducible Data Preprocessing

This notebook prepares a deliberately interpretable subset of Home Credit features for later XGBoost, SHAP, LIME, and DiCE research. It does not train or explain a model. Transformations are explicit, fitted only on training data, and leave the raw dataset unchanged.

In [1]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42

## 1. Load Data

In [2]:
DATA_PATH = Path("../data/raw/application_train.csv")
TARGET = "TARGET"
PRIMARY_FEATURES = [
    "NAME_CONTRACT_TYPE",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "DAYS_EMPLOYED",
    "NAME_INCOME_TYPE",
    "NAME_HOUSING_TYPE",
    "CNT_FAM_MEMBERS",
    "AMT_REQ_CREDIT_BUREAU_MON",
    "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
]

raw_df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {raw_df.shape}")
print("TARGET distribution:")
display(raw_df[TARGET].value_counts().sort_index())
print(f"Selected feature count: {len(PRIMARY_FEATURES)}")

Dataset shape: (307511, 122)
TARGET distribution:


TARGET
0    282686
1     24825
Name: count, dtype: int64

Selected feature count: 12


## 2. Validate Selected Features

Validation occurs before creating the independent working copy. The raw DataFrame is retained unchanged.

In [3]:
required_columns = PRIMARY_FEATURES + [TARGET]
missing_columns = sorted(set(required_columns).difference(raw_df.columns))
if missing_columns:
    raise ValueError(f"Required columns are missing from the dataset: {missing_columns}")

selected_missing_counts = raw_df[PRIMARY_FEATURES].isna().sum()
selected_feature_audit = pd.DataFrame({
    "dtype": raw_df[PRIMARY_FEATURES].dtypes.astype(str),
    "missing_count": selected_missing_counts,
    "missing_percentage": selected_missing_counts.div(len(raw_df)).mul(100),
    "unique_values": raw_df[PRIMARY_FEATURES].nunique(dropna=True),
})
display(selected_feature_audit)

working_df = raw_df[required_columns].copy()

,dtype,missing_count,missing_percentage,unique_values
NAME_CONTRACT_TYPE,object,0,0.000000,2
AMT_INCOME_TOTAL,float64,0,0.000000,2548
AMT_CREDIT,float64,0,0.000000,5603
AMT_ANNUITY,float64,12,0.003902,13672
AMT_GOODS_PRICE,float64,278,0.090403,1002
DAYS_EMPLOYED,int64,0,0.000000,12574
NAME_INCOME_TYPE,object,0,0.000000,8
NAME_HOUSING_TYPE,object,0,0.000000,6
CNT_FAM_MEMBERS,float64,2,0.000650,17
AMT_REQ_CREDIT_BUREAU_MON,float64,41519,13.501631,24


## 3. Investigate and Transform `DAYS_EMPLOYED`

In [4]:
employment_days = working_df["DAYS_EMPLOYED"]
sentinel_value = 365243
sentinel_count = employment_days.eq(sentinel_value).sum()
sentinel_percentage = sentinel_count / len(working_df) * 100

print(f"Minimum DAYS_EMPLOYED: {employment_days.min()}")
print(f"Maximum DAYS_EMPLOYED: {employment_days.max()}")
print("Most frequent DAYS_EMPLOYED values:")
display(employment_days.value_counts(dropna=False).head(10))
print(f"Rows where DAYS_EMPLOYED == {sentinel_value}: {sentinel_count}")
print(f"Percentage with sentinel value: {sentinel_percentage:.2f}%")

Minimum DAYS_EMPLOYED: -17912
Maximum DAYS_EMPLOYED: 365243
Most frequent DAYS_EMPLOYED values:


DAYS_EMPLOYED
 365243    55374
-200         156
-224         152
-230         151
-199         151
-212         150
-384         143
-229         143
-231         140
-207         138
Name: count, dtype: int64

Rows where DAYS_EMPLOYED == 365243: 55374
Percentage with sentinel value: 18.01%


The value `365243` is treated as a dataset sentinel rather than a genuine employment duration. Valid durations are converted from absolute days to years for a more understandable feature. `DAYS_EMPLOYED` is removed only from the working modelling copy; it remains unchanged in `raw_df` and in the source CSV.

In [5]:
clean_employment_days = employment_days.mask(employment_days.eq(sentinel_value), np.nan)
working_df["EMPLOYMENT_YEARS"] = clean_employment_days.abs().div(365.25)
working_df = working_df.drop(columns=["DAYS_EMPLOYED"])
working_df[["EMPLOYMENT_YEARS"]].describe()

,EMPLOYMENT_YEARS
count,252137.000000
mean,6.527500
std,6.402081
min,0.000000
25%,2.099932
50%,4.511978
75%,8.692676
max,49.040383


## 4. Create Interpretable Financial Ratios

The derived ratios provide human-readable affordability and repayment-commitment context while retaining the original financial variables. Zero denominators are treated as undefined, and infinite results are converted to missing values for later imputation.

In [6]:
numeric_before_cleaning = working_df.select_dtypes(include=np.number)
infinite_before_cleaning = np.isinf(numeric_before_cleaning.to_numpy()).sum()

def safe_ratio(numerator, denominator):
    valid_denominator = denominator.mask(denominator.eq(0), np.nan)
    return numerator.div(valid_denominator).replace([np.inf, -np.inf], np.nan)

working_df["CREDIT_INCOME_RATIO"] = safe_ratio(
    working_df["AMT_CREDIT"], working_df["AMT_INCOME_TOTAL"]
)
working_df["ANNUITY_INCOME_RATIO"] = safe_ratio(
    working_df["AMT_ANNUITY"], working_df["AMT_INCOME_TOTAL"]
)
working_df["CREDIT_ANNUITY_RATIO"] = safe_ratio(
    working_df["AMT_CREDIT"], working_df["AMT_ANNUITY"]
)
working_df.replace([np.inf, -np.inf], np.nan, inplace=True)

numeric_after_cleaning = working_df.select_dtypes(include=np.number)
infinite_after_cleaning = np.isinf(numeric_after_cleaning.to_numpy()).sum()
print(f"Infinite values before cleaning: {infinite_before_cleaning}")
print(f"Infinite values after cleaning: {infinite_after_cleaning}")

Infinite values before cleaning: 0
Infinite values after cleaning: 0


## 5. Missing Values and Feature Types After Transformation

In [7]:
modelling_feature_names = working_df.columns.drop(TARGET).tolist()
transformed_missing_counts = working_df[modelling_feature_names].isna().sum()
transformed_missing_summary = pd.DataFrame({
    "missing_count": transformed_missing_counts,
    "missing_percentage": transformed_missing_counts.div(len(working_df)).mul(100),
}).sort_values("missing_percentage", ascending=False)
display(transformed_missing_summary)

NUMERIC_FEATURES = working_df[modelling_feature_names].select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_FEATURES = working_df[modelling_feature_names].select_dtypes(exclude=np.number).columns.tolist()
print("NUMERIC_FEATURES:")
print(NUMERIC_FEATURES)
print("CATEGORICAL_FEATURES:")
print(CATEGORICAL_FEATURES)

,missing_count,missing_percentage
EMPLOYMENT_YEARS,55374,18.007161
AMT_REQ_CREDIT_BUREAU_MON,41519,13.501631
AMT_REQ_CREDIT_BUREAU_YEAR,41519,13.501631
AMT_REQ_CREDIT_BUREAU_QRT,41519,13.501631
AMT_GOODS_PRICE,278,0.090403
AMT_ANNUITY,12,0.003902
CREDIT_ANNUITY_RATIO,12,0.003902
ANNUITY_INCOME_RATIO,12,0.003902
CNT_FAM_MEMBERS,2,0.000650
NAME_HOUSING_TYPE,0,0.000000


NUMERIC_FEATURES:
['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_FAM_MEMBERS', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'EMPLOYMENT_YEARS', 'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'CREDIT_ANNUITY_RATIO']
CATEGORICAL_FEATURES:
['NAME_CONTRACT_TYPE', 'NAME_INCOME_TYPE', 'NAME_HOUSING_TYPE']


## 6. Train/Test Split Before Fitting Preprocessors

The stratified split is performed before any imputer or encoder is fitted. This prevents information from the held-out test partition influencing learned preprocessing parameters.

In [8]:
X = working_df[modelling_feature_names].copy()
y = working_df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print("y_train TARGET distribution:")
display(y_train.value_counts().sort_index())
print("y_test TARGET distribution:")
display(y_test.value_counts().sort_index())

X_train shape: (246008, 15)
X_test shape: (61503, 15)
y_train TARGET distribution:


TARGET
0    226148
1     19860
Name: count, dtype: int64

y_test TARGET distribution:


TARGET
0    56538
1     4965
Name: count, dtype: int64

## 7. Build and Fit the Preprocessing Pipeline

Numerical missing values use the training median. Categorical missing values use the training mode before one-hot encoding with unknown test categories ignored. Numerical scaling is intentionally omitted because tree-based XGBoost models do not require standardisation.

In [9]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## 8. Inspect Processed Output

In [10]:
def count_missing_processed(matrix):
    values = matrix.data if hasattr(matrix, "data") and not isinstance(matrix, np.ndarray) else np.asarray(matrix)
    return int(np.isnan(values).sum())

train_missing_after_preprocessing = count_missing_processed(X_train_processed)
test_missing_after_preprocessing = count_missing_processed(X_test_processed)
transformed_feature_names = preprocessor.get_feature_names_out()
transformed_feature_series = pd.Series(transformed_feature_names, name="transformed_feature")

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")
print(f"Missing values in processed training data: {train_missing_after_preprocessing}")
print(f"Missing values in processed test data: {test_missing_after_preprocessing}")
print(f"Total transformed feature count: {len(transformed_feature_names)}")
display(transformed_feature_series.head(50).to_frame())

Processed training shape: (246008, 28)
Processed test shape: (61503, 28)
Missing values in processed training data: 0
Missing values in processed test data: 0
Total transformed feature count: 28


,transformed_feature
0,numeric__AMT_INCOME_TOTAL
1,numeric__AMT_CREDIT
2,numeric__AMT_ANNUITY
3,numeric__AMT_GOODS_PRICE
4,numeric__CNT_FAM_MEMBERS
5,numeric__AMT_REQ_CREDIT_BUREAU_MON
6,numeric__AMT_REQ_CREDIT_BUREAU_QRT
7,numeric__AMT_REQ_CREDIT_BUREAU_YEAR
8,numeric__EMPLOYMENT_YEARS
9,numeric__CREDIT_INCOME_RATIO


## 9. Preserve Interpretability Metadata

This mapping is a research control for later explanation and counterfactual work, not a legal classification. Partially actionable features remain locked pending manual DiCE review. Derived ratios must be recalculated from underlying features rather than manipulated independently.

In [11]:
categorical_encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
categorical_original_features = [
    feature
    for feature, categories in zip(CATEGORICAL_FEATURES, categorical_encoder.categories_)
    for _ in categories
]
transformed_to_original = NUMERIC_FEATURES + categorical_original_features
assert len(transformed_to_original) == len(transformed_feature_names)

actionable_features = {"AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"}
partially_actionable_features = {
    "AMT_INCOME_TOTAL", "EMPLOYMENT_YEARS", "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO",
    "AMT_REQ_CREDIT_BUREAU_MON", "AMT_REQ_CREDIT_BUREAU_QRT",
    "AMT_REQ_CREDIT_BUREAU_YEAR",
}
locked_features = {
    "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE",
    "CNT_FAM_MEMBERS",
}
derived_ratio_features = {
    "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_ANNUITY_RATIO",
}

def actionability_for(feature):
    if feature in actionable_features:
        return "Actionable"
    if feature in partially_actionable_features:
        return "Partially actionable"
    if feature in locked_features:
        return "Non-actionable / locked"
    return "Review"

def feature_type_for(feature):
    if feature in derived_ratio_features:
        return "Derived numerical"
    if feature in NUMERIC_FEATURES:
        return "Numerical"
    return "Categorical"

preprocessing_metadata = pd.DataFrame({
    "original_feature": transformed_to_original,
    "transformed_feature": transformed_feature_names,
})
preprocessing_metadata["feature_type"] = preprocessing_metadata["original_feature"].map(feature_type_for)
preprocessing_metadata["actionability"] = preprocessing_metadata["original_feature"].map(actionability_for)
preprocessing_metadata["allowed_for_dice"] = preprocessing_metadata["original_feature"].isin(actionable_features)
preprocessing_metadata.loc[
    preprocessing_metadata["original_feature"].isin(derived_ratio_features), "allowed_for_dice"
] = False
display(preprocessing_metadata)

,original_feature,transformed_feature,feature_type,actionability,allowed_for_dice
0,AMT_INCOME_TOTAL,numeric__AMT_INCOME_TOTAL,Numerical,Partially actionable,False
1,AMT_CREDIT,numeric__AMT_CREDIT,Numerical,Actionable,True
2,AMT_ANNUITY,numeric__AMT_ANNUITY,Numerical,Actionable,True
3,AMT_GOODS_PRICE,numeric__AMT_GOODS_PRICE,Numerical,Actionable,True
4,CNT_FAM_MEMBERS,numeric__CNT_FAM_MEMBERS,Numerical,Non-actionable / locked,False
5,AMT_REQ_CREDIT_BUREAU_MON,numeric__AMT_REQ_CREDIT_BUREAU_MON,Numerical,Partially actionable,False
6,AMT_REQ_CREDIT_BUREAU_QRT,numeric__AMT_REQ_CREDIT_BUREAU_QRT,Numerical,Partially actionable,False
7,AMT_REQ_CREDIT_BUREAU_YEAR,numeric__AMT_REQ_CREDIT_BUREAU_YEAR,Numerical,Partially actionable,False
8,EMPLOYMENT_YEARS,numeric__EMPLOYMENT_YEARS,Numerical,Partially actionable,False
9,CREDIT_INCOME_RATIO,numeric__CREDIT_INCOME_RATIO,Derived numerical,Partially actionable,False


## 10. Quality Checks

In [12]:
duplicate_rows = working_df.duplicated().sum()
train_class_balance = y_train.value_counts(normalize=True).sort_index().mul(100)
test_class_balance = y_test.value_counts(normalize=True).sort_index().mul(100)

print(f"Duplicate rows in selected working data: {duplicate_rows}")
print(f"Infinite values before cleaning: {infinite_before_cleaning}")
print(f"Infinite values after cleaning: {infinite_after_cleaning}")
print(f"Remaining NaN in processed training data: {train_missing_after_preprocessing}")
print(f"Remaining NaN in processed test data: {test_missing_after_preprocessing}")
print("Training class balance (%):")
display(train_class_balance)
print("Test class balance (%):")
display(test_class_balance)
print(f"Transformed training matrix: {X_train_processed.shape}")
print(f"Transformed test matrix: {X_test_processed.shape}")

assert infinite_after_cleaning == 0, "Infinite values remain after cleaning."
assert train_missing_after_preprocessing == 0, "NaN values remain in processed training data."
assert test_missing_after_preprocessing == 0, "NaN values remain in processed test data."
assert X_train_processed.shape[1] == X_test_processed.shape[1]
assert len(transformed_feature_names) == X_train_processed.shape[1]
assert len(X_train) + len(X_test) == len(working_df)

Duplicate rows in selected working data: 2644
Infinite values before cleaning: 0
Infinite values after cleaning: 0
Remaining NaN in processed training data: 0
Remaining NaN in processed test data: 0
Training class balance (%):


TARGET
0    91.927092
1     8.072908
Name: proportion, dtype: float64

Test class balance (%):


TARGET
0    91.927223
1     8.072777
Name: proportion, dtype: float64

Transformed training matrix: (246008, 28)
Transformed test matrix: (61503, 28)


## 11. Save Preprocessing Artifacts

Executing this cell saves the fitted preprocessing pipeline and non-identifying feature metadata. It does not save the raw, training, test, or full transformed datasets.

In [13]:
ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
PREPROCESSOR_PATH = ARTIFACTS_DIR / "preprocessor.joblib"
METADATA_PATH = ARTIFACTS_DIR / "preprocessing_metadata.csv"
PRIMARY_FEATURES_PATH = ARTIFACTS_DIR / "primary_features.json"

joblib.dump(preprocessor, PREPROCESSOR_PATH)
preprocessing_metadata.to_csv(METADATA_PATH, index=False)
with PRIMARY_FEATURES_PATH.open("w", encoding="utf-8") as file:
    json.dump(modelling_feature_names, file, indent=2)

print(f"Preprocessor saved: {PREPROCESSOR_PATH}")
print(f"Metadata saved: {METADATA_PATH}")
print(f"Final original modelling features saved: {PRIMARY_FEATURES_PATH}")

Preprocessor saved: ..\artifacts\preprocessor.joblib
Metadata saved: ..\artifacts\preprocessing_metadata.csv
Final original modelling features saved: ..\artifacts\primary_features.json


## 12. Optional Compact Processed Sample

In [14]:
sample_size = min(10, X_test_processed.shape[0])
sample_values = X_test_processed[:sample_size]
if hasattr(sample_values, "toarray"):
    sample_values = sample_values.toarray()
processed_sample = pd.DataFrame(sample_values, columns=transformed_feature_names)
display(processed_sample)

,numeric__AMT_INCOME_TOTAL,numeric__AMT_CREDIT,numeric__AMT_ANNUITY,numeric__AMT_GOODS_PRICE,numeric__CNT_FAM_MEMBERS,numeric__AMT_REQ_CREDIT_BUREAU_MON,numeric__AMT_REQ_CREDIT_BUREAU_QRT,numeric__AMT_REQ_CREDIT_BUREAU_YEAR,numeric__EMPLOYMENT_YEARS,numeric__CREDIT_INCOME_RATIO,...,categorical__NAME_INCOME_TYPE_State servant,categorical__NAME_INCOME_TYPE_Student,categorical__NAME_INCOME_TYPE_Unemployed,categorical__NAME_INCOME_TYPE_Working,categorical__NAME_HOUSING_TYPE_Co-op apartment,categorical__NAME_HOUSING_TYPE_House / apartment,categorical__NAME_HOUSING_TYPE_Municipal apartment,categorical__NAME_HOUSING_TYPE_Office apartment,categorical__NAME_HOUSING_TYPE_Rented apartment,categorical__NAME_HOUSING_TYPE_With parents
0,157500.0,770292.0,30676.5,688500.0,3.0,0.0,1.0,2.0,0.287474,4.890743,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
1,90000.0,364896.0,19926.0,315000.0,2.0,1.0,0.0,6.0,13.497604,4.054400,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,148500.0,284400.0,18643.5,225000.0,2.0,0.0,0.0,3.0,3.260780,1.915152,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,188100.0,976711.5,38218.5,873000.0,1.0,0.0,0.0,1.0,0.971937,5.192512,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,180000.0,323194.5,19660.5,279000.0,2.0,0.0,0.0,1.0,3.739904,1.795525,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
5,90000.0,318528.0,8532.0,252000.0,3.0,1.0,0.0,0.0,1.234771,3.539200,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
6,319500.0,1431000.0,39483.0,1431000.0,2.0,1.0,0.0,1.0,15.696099,4.478873,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7,157500.0,382500.0,19125.0,382500.0,2.0,0.0,0.0,1.0,4.511978,2.428571,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
8,112500.0,170640.0,13612.5,135000.0,1.0,0.0,0.0,3.0,1.869952,1.516800,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
9,202500.0,225000.0,11250.0,225000.0,1.0,0.0,0.0,2.0,4.511978,1.111111,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


## Preprocessing Decisions

- The source CSV and `raw_df` remain unchanged; all transformations operate on an explicit working copy.
- The primary subset prioritises features that can support understandable credit-decision explanations.
- `DAYS_EMPLOYED` uses a known sentinel value, so valid durations are converted to the more interpretable `EMPLOYMENT_YEARS`.
- Numerical missing values are imputed with training medians, while categorical missing values use training modes.
- Categorical values are one-hot encoded, and previously unseen test categories are safely ignored.
- The preprocessor is fitted only on training data to prevent test information leaking into learned transformations.
- Numerical scaling is omitted because the intended tree-based XGBoost model does not require standardisation.
- Actionability metadata distinguishes directly changeable, review-only, locked, and derived features for later DiCE controls.
- `random_state = 42` makes the stratified split reproducible.

These are methodology decisions for subsequent evaluation, not final dissertation or legal conclusions.

## 14. Final Verification

In [15]:
print(f"Raw rows: {len(raw_df)}")
print(f"Selected original features: {len(PRIMARY_FEATURES)}")
print(f"Final modelling features before encoding: {len(modelling_feature_names)}")
print(f"Numerical features: {len(NUMERIC_FEATURES)}")
print(f"Categorical features: {len(CATEGORICAL_FEATURES)}")
print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Processed feature count: {len(transformed_feature_names)}")
print(f"Remaining NaN after preprocessing: {train_missing_after_preprocessing + test_missing_after_preprocessing}")
print(f"Preprocessor saved: {PREPROCESSOR_PATH.exists()} ({PREPROCESSOR_PATH})")
print(f"Metadata saved: {METADATA_PATH.exists()} ({METADATA_PATH})")

Raw rows: 307511
Selected original features: 12
Final modelling features before encoding: 15
Numerical features: 12
Categorical features: 3
Train rows: 246008
Test rows: 61503
Processed feature count: 28
Remaining NaN after preprocessing: 0
Preprocessor saved: True (..\artifacts\preprocessor.joblib)
Metadata saved: True (..\artifacts\preprocessing_metadata.csv)
